In [0]:
delta_supplychain_table = spark.read.csv("/Volumes/workspace/supplychain_bigdata/supply/DataCoSupplyChainDataset.csv"
               ,header=True
               ,inferSchema=True)


In [0]:
spark.table("supplychain_table").columns

In [0]:
description_table = spark.read.csv("/Volumes/workspace/supplychain_bigdata/supply/DescriptionDataCoSupplyChain.csv"
               ,header=True
               ,inferSchema=True)

description_table.write.format("delta").mode("overwrite").option("delta.columnMapping.mode", "name").saveAsTable("description_table")

spark.table("description_table").columns

In [0]:
token_table = spark.read.csv("/Volumes/workspace/supplychain_bigdata/supply/tokenized_access_logs.csv",
                             header=True,
                             inferSchema=True)

token_table.write.format("delta").mode("overwrite").option("delta.columnMapping.mode", "name").saveAsTable("token_table")
spark.table("token_table").columns

In [0]:
%sql
select * from token_table

In [0]:
%sql
select * from supplychain_table limit 5

In [0]:
import re

clean_col = [
        re.sub(r"[ ,;{}()\n\t=]+", "_", col.strip()) for col in spark.table("supplychain_table").columns
]
supply_chain_clean = spark.table("supplychain_table").toDF(*clean_col)


In [0]:
supply_chain_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("supplychain_table")

In [0]:
clean_col = [
    re.sub(r"[ ,;{}()\n\t=]+", "_", col.strip()).rstrip("_") for col in spark.table("supplychain_table").columns
]
supply_chain_clean = spark.table("supplychain_table").toDF(*clean_col)
supply_chain_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("supplychain_table")
spark.table("supplychain_table").dtypes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col

tb_update = spark.table("supplychain_table")\
    .withColumn(
    "order_date_DateOrders", 
    to_timestamp(col("order_date_DateOrders"), "M/d/yyyy H:mm").cast("date")
)\
    .withColumn("shipping_date_DateOrders", to_timestamp(col("order_date_DateOrders"), "M/d/yyyy H:mm").cast("date"))

tb_update.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("supplychain_table")
spark.table("supplychain_table").dtypes